# Machine Learning: From Simple Classifier to Variational Autoencoder

Today we'll embark on a journey through deep learning using the famous MNIST dataset. We'll start with a simple feedforward neural network and then advance to a sophisticated Variational Autoencoder (VAE). Let's dive in!

The MNIST dataset consists of 70,000 grayscale images (28×28 pixels) of handwritten digits (0-9). It's the "Hello World" of deep learning. Each image has:
* Input: 784 pixel values (28 × 28 flattened) with values between 0 and 1
* Label: A digit from 0 to 9

In [ ]:
%load_ext autoreload
%autoreload 2

import mlkit as mlk

from pathlib import Path  # Gives us a clear, cross-platform way to name folders.

import torch
from torch import nn # The neural-network building blocks, such as Linear and ReLU.

from torchvision import datasets, transforms # Provides MNIST and image preprocessing.
from torch.utils.data import DataLoader # Creates mini-batches from a dataset.

## Functions

Function 1: Set device

Use the fastest computing resource available to run your model

The function executes a strict hierarchy of checks:
* Is there an NVIDIA GPU? -> YES: Use cuda (Fastest)
* Is there an Apple Silicon GPU? -> YES: Use mps (Very Fast on Mac)
* Otherwise? -> Use cpu (Guaranteed to work)

In [ ]:
# Use the fastest computing resource available to run your model
def choose_device() -> torch.device:
    """Choose the fastest available device that PyTorch can use."""
    if torch.cuda.is_available():  # CUDA means an NVIDIA GPU is available.
        return torch.device("cuda")  # Put tensors and model weights on the NVIDIA GPU.
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():  # Apple GPU check.
        return torch.device("mps")  # Use Apple's Metal backend on supported Macs.
    return torch.device("cpu")  # Fall back to the CPU, which works everywhere.

Function 2: Downloading MNIST dataset

Function is responsible for the entire data pipeline setup for a standard image classification task using MNIST. 

It handles downloading, preprocessing, and batching the data.

Code for transform:
* `transforms.Compose()`: container that chains multiple transformations together, executing them one after the other (sequential)
* `transforms.ToTensor()`: MNIST images are initially loaded as standard image objects. This function converts that image object into a PyTorch tensor. Specifically, it scales the pixel values form the standard range [0, 255] down to floating numbers in the range of [0.0, 1.0] (Pixel Value Normalization). 
    * Pixel Value Normalization: the transformation effectively performs the division: New Value = Original Pixel Value/255
    * Black pixel (value 0) becomes 0/255 = 0.0
    * White pixel (value 255) becomes 255/255 = 1.0
    * Mid-gray pixel (e.g value 127) becomes 127/255 = 0.498
    * Why is this Pixel Value Normalization important? NN rely on mathematical functions (like Sigmoid, Tanh, or ReLU) that perform best and most stably when the input data is in a small, standardized range
        * **Numerical stability**: If you were to use input values in the [0,255] range, these numbers would be very large. When these large numbers are multiplied by the network's weights during training, the intermediate results can become extremely large. This leads to numerical overflow or highly unstable gradients during training.
        * **Convergence speed**: when the input data is scaled to [0,1], the optimizer (e.g. Adam or SGD) can adjust the network weights much more efficiently and quickly because it is not dealing with massive numbers

Code for dataset:
* `root=DATA_DIR`: specifies where the dataset files (images and labels) should be stored
* `train=True/train=False`: tells the function whether to load the 60,000 images designated for training or the 10,000 images reserved for testing

In [ ]:
# Downlading MNIST dataset
def dataloader(device: torch.device) -> tuple[DataLoader, DataLoader]:
    """Download MNIST and wrap the train and test splits in DataLoaders."""
    transform = transforms.Compose(  # Compose runs preprocessing steps in order.
        [
            transforms.ToTensor(),  # Converts a PIL image to a tensor with values in [0, 1].
            transforms.Normalize((0.1307,), (0.3081,)),  # Standard MNIST mean and std scaling.
        ]
    )
    train_dataset = datasets.MNIST(  # The supervised training split contains 60,000 images.
        root=DATA_DIR,  # Store or read MNIST files from DATA_DIR.
        train=True,  # Select the training split.
        download=True,  # Download the dataset automatically if it is missing.
        transform=transform,  # Apply tensor conversion and normalization to each image.
    )
    test_dataset = datasets.MNIST(  # The test split contains 10,000 held-out images.
        root=DATA_DIR,  # Use the same cache folder as the training dataset.
        train=False,  # Select the test split.
        download=True,  # Reuse files if present, otherwise download them.
        transform=transform,  # Apply the exact same preprocessing at test time.
    )
    train_loader = DataLoader(  # The loader creates shuffled mini-batches for training.
        train_dataset,  # The dataset that supplies images and labels.
        batch_size=BATCH_SIZE,  # Number of examples per mini-batch.
        shuffle=True,  # Shuffle training data so batches vary across epochs.
        pin_memory=device.type == "cuda",  # Speeds CPU-to-GPU copies when using CUDA.
    )
    test_loader = DataLoader(  # The test loader creates mini-batches for evaluation.
        test_dataset,  # The held-out dataset used to measure generalization.
        batch_size=BATCH_SIZE,  # Evaluation can use the same batch size.
        shuffle=False,  # Keep test order stable because evaluation does not learn.
        pin_memory=device.type == "cuda",  # This only matters on CUDA.
    )
    return train_loader, test_loader  # Give both loaders back to the caller.

# 1. Simple Neural Network Classifier

Why Use a convolutional neural network (CNN)?

A fully connected network can classify MNIST, but a CNN is more natural for images. A convolutional layer learns small filters such as edge detectors, stroke detectors, and curve detectors. The same filter is reused across the image, so the model learns spatial patterns efficiently.

The classifier uses:

image -> conv -> ReLU -> pool -> conv -> ReLU -> pool -> flatten -> linear -> ReLU -> linear -> logits


Training Logic

Each mini-batch follows the standard PyTorch training recipe:
1. move data to device
2. clear old gradients
3. run forward pass
4. compute loss
5. run backward pass
6. update weights
7. record metrics

The model learns because `loss.backward()` computes gradients, and `optimizer.step()` uses those gradients to adjust the parameters.

## 1.1 Setup

In [22]:
# batch size of 32 performs best cf. REVISITING SMALL BATCH TRAINING FOR DEEP NEURAL NETWORKS
BATCH_SIZE = 32  # The model sees 64 images before each weight update.

In [31]:
# Set the device
device = choose_device()  # Decide whether to use CUDA, Apple MPS, or CPU.
device

device(type='mps')

## 1.2 Loading data

Transformation: We need to 

In [29]:
DATA_DIR = Path("data")  # MNIST will be downloaded and cached in this local folder.

In [ ]:
# Downloading the MNIST dataset
train_loader, test_loader = dataloader(device)  # Prepare MNIST mini-batches.

100%|██████████| 9.91M/9.91M [00:03<00:00, 3.17MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 227kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 1.66MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.57MB/s]
